# Gatefall — ComfyUI on Google Colab (free GPU)

Free way to generate Gatefall character art without a local GPU. Runs
ComfyUI on Colab's free Tesla T4, downloads an anime checkpoint
(Illustrious or Pony Diffusion XL), and exposes the ComfyUI web UI in
your browser via Colab's own port proxy (with a Cloudflare tunnel as
fallback).

**Before running:** `Runtime` menu -> `Change runtime type` -> select
`T4 GPU`, then `Save`. Run the cells below in order.

See `docs/art-direction.md` for the character prompts to paste in, and
`docs/comfyui-tutorial.md` for the full local-install version of this
guide.

**Colab free-tier limits:** sessions disconnect after inactivity and
have a rolling usage cap. Save any image you want to keep — closing
the tab or losing the session wipes the Colab disk.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi

## 2. Install ComfyUI

In [ ]:
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -r requirements.txt -q

## 3. Get a checkpoint (Illustrious or Pony Diffusion XL)

Two ways to get the checkpoint into Colab — pick **one** of 3a / 3b
and run only that cell.

### 3a. Download directly into Colab from Civitai

1. Go to civitai.com and search "Illustrious XL" or "Pony Diffusion
   XL", open the base checkpoint model page.
2. Right-click the **Download** button -> copy link address
   (looks like `https://civitai.com/api/download/models/XXXXXX`).
3. Paste it into `CHECKPOINT_URL` below.

Some Civitai models require being logged in to download. If the
download fails with an auth error, get an API key from
civitai.com -> account settings -> API Keys, and paste it into
`CIVITAI_TOKEN`. Leave it as an empty string if not needed.

In [ ]:
CHECKPOINT_URL = "https://civitai.com/api/download/models/REPLACE_ME"  # @param {type:"string"}
CIVITAI_TOKEN = ""  # @param {type:"string"}

import os
url = CHECKPOINT_URL
if CIVITAI_TOKEN:
    sep = "&" if "?" in url else "?"
    url = f"{url}{sep}token={CIVITAI_TOKEN}"

os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
!wget -nc --content-disposition "$url" -P /content/ComfyUI/models/checkpoints
!ls -lh /content/ComfyUI/models/checkpoints

### 3b. Already downloaded the checkpoint to your computer? Use Google Drive

Re-uploading a multi-GB file straight into a Colab cell (e.g. via the
Files panel's upload button) is slow and gets wiped when the session
ends. Instead, upload it to Drive once and mount Drive here — it
persists across sessions, so you only upload it one time.

1. On your computer, upload the `.safetensors` file to your Google
   Drive (e.g. into a folder called `gatefall-checkpoints`) — via
   drive.google.com in a browser, or drag-and-drop into the Drive
   desktop app. This step happens outside Colab and can take a while
   for a multi-GB file, but it's one-time.
2. Run the cell below — it mounts your Drive (you'll get a browser
   popup asking to authorize access) and copies the file into
   ComfyUI's checkpoints folder.
3. Update `DRIVE_CHECKPOINT_PATH` to match where you put the file.

In [ ]:
DRIVE_CHECKPOINT_PATH = "/content/drive/MyDrive/gatefall-checkpoints/ponyDiffusionV6XL.safetensors"  # @param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")

import os, shutil
os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
dest = os.path.join("/content/ComfyUI/models/checkpoints", os.path.basename(DRIVE_CHECKPOINT_PATH))
# Symlink instead of copy: instant, and avoids duplicating a multi-GB file onto the Colab disk.
if not os.path.exists(dest):
    os.symlink(DRIVE_CHECKPOINT_PATH, dest)
!ls -lh /content/ComfyUI/models/checkpoints

Either way, confirm a `.safetensors` file of several GB shows up in the
listing above before continuing (3a: bad URL/missing token; 3b: wrong
`DRIVE_CHECKPOINT_PATH` or Drive not finished uploading, are the usual
culprits).

## 4. Launch ComfyUI and open it in your browser

### 4a. Colab's built-in port proxy (recommended, try this first)

Uses Google's own infrastructure instead of a third-party tunnel — no
separate service to fail, and it's authenticated to your Google
account automatically. This avoids the `HTTP ERROR 403 — Access
denied` page that Cloudflare's free `trycloudflare.com` quick tunnels
sometimes throw (some ISPs/networks get blocked at Cloudflare's edge,
unrelated to anything you did).

In [ ]:
%cd /content/ComfyUI

import subprocess, time

comfy_proc = subprocess.Popen(
    ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
)
time.sleep(15)  # give ComfyUI time to start before opening the proxy

from google.colab.output import eval_js
proxy_url = eval_js("google.colab.kernel.proxyPort(8188)")
print(f"Open ComfyUI here: {proxy_url}")

Click the printed `https://8188-....colab.googleusercontent.com/`
link — that's your ComfyUI web UI, same interface as a local install.
Only usable while signed into the same Google account viewing this
notebook.

**This cell keeps running** (it's what keeps ComfyUI alive) — leave it
running while you work, don't interrupt it until you're done for the
session.

### 4b. Fallback: Cloudflare quick tunnel

Only use this if 4a doesn't work for some reason. Run **either** 4a or
4b, not both (both try to launch ComfyUI on the same port — restart
the runtime first if you already ran 4a).

In [ ]:
%cd /content/ComfyUI
!wget -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

import subprocess, time, re

comfy_proc = subprocess.Popen(
    ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
)
time.sleep(15)  # give ComfyUI time to start before opening the tunnel

tunnel_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Waiting for tunnel URL...")
for line in tunnel_proc.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print(f"\n\nOpen ComfyUI here: {match.group(0)}\n")
        break

If this link 403s (`Access denied — You don't have authorization to
view this page`), that's Cloudflare's edge blocking the quick-tunnel
domain on your network — go back and use 4a instead, or try re-running
this cell to get a fresh random subdomain.

## 5. Generate

In the ComfyUI web UI:
1. Default graph already has `Load Checkpoint` -> prompts ->
   `KSampler` -> `Save Image`. In `Load Checkpoint`, select the file
   you added in step 3.
2. Paste a positive/negative prompt from `docs/art-direction.md`
   (e.g. Faelen's SD/Illustrious draft) into the two `CLIP Text
   Encode` boxes. If you're on Pony Diffusion V6 XL, prefix the
   positive prompt with its expected quality tags:
   `score_9, score_8_up, score_7_up, ` before the rest of the tags.
3. Use an SDXL-native resolution (1024x1024, or 832x1216 for a
   portrait model sheet) in the `Empty Latent Image` node.
4. Click **Queue Prompt**.

Generated images save to `/content/ComfyUI/output/` on the Colab VM —
download anything you want to keep before the session ends (files
panel on the left, or `!zip` + download the archive).

## 6. (Optional) Download all outputs as a zip

In [ ]:
from google.colab import files
!zip -r /content/gatefall_outputs.zip /content/ComfyUI/output
files.download("/content/gatefall_outputs.zip")